# Δημιουργία BERT Embeddings

Στο notebook [σημασιολογικής αναζήτησης](https://colab.research.google.com/drive/19zJiRPA4IqZH9oF34Fc77W_JoBv0ONMv?usp=sharing) χρησιμοποιήσαμε τις εντολές `SentenceTransformer('bert-base-uncased')` από την Hugging Face, και καλέσαμε `model.encode(text)`. Αυτές οι εντολές χειρίζονται τα πάντα αθόρυβα: tokenization, το forward pass του transformer, και το **pooling** (τον μέσο όρο) των συμφραζόμενων διανυσμάτων των τελικών tokens σε ένα ενιαίο διάνυσμα πρότασης.

Σε αυτό το notebook ανοίγουμε αυτό το μαύρο κουτί:

1. Θα φορτώσουμε απευθείας τα βάρη BERT με `AutoModelForMaskedLM` από το Hugging Face Transformers.
2. Θα αφαιρέσουμε το classificaiton head (MLM head) ώστε να κρατήσουμε μόνο τον transformer encoder.
3. Θα κάνουμε tokenize το κείμενο μόνοι μας με `AutoTokenizer`.
4. Θα εκτελέσουμε το forward pass και κάνουμε **mean-pool** τα token embeddings (θα υπολογίσουμε τον μέσο όρο).
5. Θα υπολογίσουμε την ενσωμάτωση του ίδιου κειμένου και με τους δύο τρόπους και θα δούμε (θα επιβεβαιώσουμε!) ότι τα διανύσματα είναι πανομοιότυπα.

```
Κωνσταντίνος Καραμανής: constantine@utexas.edu
http://users.ece.utexas.edu/~cmcaram/
The University of Texas at Austin
Archimedes/Athena RC
```

## Εισαγωγή Βιβλιοθηκών

Πάλι χρησιμοποιούμε τις βιβλιοθήκες του Hugging Face. Όπως έχουμε πει και στο προηγούμενο notebook, το Hugging Face είναι μια πλατφόρμα και συγχρόνως μια κοινότητα για τεχνητή νοημοσύνη και μηχανική μάθηση. Περιλαμβάνει πολλά μοντέλα, σύνολα δεδομένων και βιβλιοθήκες όπως οι Transformers, SentenceTransformers, Tokenizers και AutoModel.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sentence_transformers import SentenceTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


## Τι είναι το `AutoModelForMaskedLM`;

Το BERT εκπαιδεύτηκε εκ των προτέρων με **Masked Language Modelling (MLM)**: τυχαία tokens καλύπτονται και το μοντέλο πρέπει να προβλέψει ποια ήταν. Το `AutoModelForMaskedLM` φορτώνει το πλήρες μοντέλο, το οποίο έχει δύο μέρη:

```
BertForMaskedLM
├── bert   ← ο transformer encoder (αυτό που θέλουμε για embeddings)
└── cls    ← το classification head (MLM head) που χρησιμοποιήθηκε για να λυθεί το
πρόβλημα ταξινόμησης του masked language modelling (προβολή μεγέθους λεξιλογίου
που δεν χρειαζόμαστε)
```

Αποκτούμε πρόσβαση μόνο στον encoder μέσω `.bert`:

```python
full_model = AutoModelForMaskedLM.from_pretrained('google-bert/bert-base-uncased')
encoder    = full_model.bert   # BertModel — χωρίς classification head
```

Αυτό είναι ακριβώς ισοδύναμο με `AutoModel.from_pretrained('google-bert/bert-base-uncased')`, απλώς γραμμένο ρητά για να φανεί πού βρίσκεται ο encoder.

<img src="https://drive.google.com/uc?id=1aOJxDjHGC032hR8QUWQ4PVc0_b06zACi" alt="BERT Embeddings" width="500" height="auto">

## Η κλάση `BertEmbedder`

Δημιουργούμε ένα PyTorch μοντέλο που υλοποιεί τα βήματα που περιέχει (σαν μαύρο κουτί) το SentenceTransformer.

### Mean pooling:

Το BERT παράγει ένα 768-διάστατο διάνυσμα για κάθε token εισόδου. Για μια (1) πρόταση μήκους $n$ tokens (~λέξεις) παράγει output σχήματος $(1, n, 768)$. Για να πάρουμε ένα ενιαίο διάνυσμα πρότασης υπολογίζουμε τον μέσο όρο ανά τη διάσταση των tokens — αλλά πρέπει να αγνοήσουμε τα tokens του padding, τα οποία δεν αντιστοιχούν σε πραγματικές λέξεις. Το `attention_mask` έχει τιμή 1 για τα πραγματικά tokens και 0 για το padding, οπότε το χρησιμοποιούμε ως βάρος.

### Γιατί χρειαζόμαστε το Padding και το Masking;

Όταν επεξεργαζόμαστε πολλές προτάσεις ταυτόχρονα (σε batches, όπως μας δίνουν οι ``data loaders``), πρέπει όλες να έχουν το ίδιο μήκος για να διευκολύνουν τους υπολογισμούς του μοντέλου. Επειδή οι προτάσεις φυσικά διαφέρουν σε μήκος, προσθέτουμε ειδικά **padding tokens** (συνήθως το `[PAD]` που είναι το token `[0]`) στις μικρότερες προτάσεις ώστε να φτάσουν το μήκος της μεγαλύτερης στο batch.

Ωστόσο, αυτά τα padding tokens δεν φέρουν καμία σημασιολογική πληροφορία και δεν θέλουμε να επηρεάσουν τους υπολογισμούς του BERT ούτε την τελική αναπαράσταση της πρότασης. Για αυτό το λόγο χρησιμοποιούμε το **attention mask**: είναι ένας πίνακας με άσσους (1) για τα πραγματικά tokens και μηδενικά (0) για τα padding tokens.

Όπως θα δείτε στον κώδικα παρακάτω (στη μέθοδο `mean_pool`), χρησιμοποιούμε το `attention_mask` ως βάρος, ώστε να μηδενίσουμε τα embeddings των padding tokens πριν τον υπολογισμό του αθροίσματος και να διαιρέσουμε μόνο με το πλήθος των *πραγματικών* λέξεων.

In [ ]:
class BertEmbedder(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        # AutoModelForMaskedLM loads BERT + MLM head.
        # .bert strips the head and keeps only the transformer encoder.
        self.bert = AutoModelForMaskedLM.from_pretrained(model_name).bert
        # self.bert = AutoModel.from_pretrained(model_name)  # equivalent

    def mean_pool(self, token_embeddings, attention_mask):
        # token_embeddings: (batch, seq_len, hidden_dim)
        # attention_mask:   (batch, seq_len)  — 1 for real tokens, 0 for padding
        mask = attention_mask.unsqueeze(-1).float()           # (batch, seq_len, 1)
        sum_embeddings = (token_embeddings * mask).sum(dim=1) # (batch, hidden_dim)
        sum_mask = mask.sum(dim=1).clamp(min=1e-9)            # (batch, 1)
        return sum_embeddings / sum_mask                       # (batch, hidden_dim)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        # outputs.last_hidden_state: (batch, seq_len, hidden_dim)
        return self.mean_pool(outputs.last_hidden_state, attention_mask)

## Φόρτωση και των Δύο Μοντέλων

Φορτώνουμε το `SentenceTransformer('bert-base-uncased')` και το δικό μας `BertEmbedder`.

In [ ]:
MODEL_NAME = 'google-bert/bert-base-uncased'

# Our hand-rolled embedder
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
our_model = BertEmbedder(MODEL_NAME).to(device).eval()

# SentenceTransformer wrapper
st_model  = SentenceTransformer('bert-base-uncased', device=str(device))

print('BertEmbedder encoder type:', type(our_model.bert))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BertEmbedder encoder type: <class 'transformers.models.bert.modeling_bert.BertModel'>


## Ένα ποίημα για το ML και το AI

Αυτό είναι το κείμενο που θα ενσωματώσουμε.

In [ ]:
poem = """
A model learns from data, line by line,
adjusting weights until the patterns shine.
Gradients descend through layers deep,
while loss curves fall and neurons keep.

Attention heads scan left and right,
encoding meaning, context, light.
From tokens sparse to vectors dense,
the network builds its own good sense.

Not magic — just arithmetic at scale,
yet language blooms beyond the pale.
"""

print(poem)


A model learns from data, line by line,
adjusting weights until the patterns shine.
Gradients descend through layers deep,
while loss curves fall and neurons keep.

Attention heads scan left and right,
encoding meaning, context, light.
From tokens sparse to vectors dense,
the network builds its own good sense.

Not magic — just arithmetic at scale,
yet language blooms beyond the pale.



### O Tokenizer

Γιατί δεν χρησιμοποιούμε ``collate`` εδώ;


(Δείτε [το προηγούμενο Notebook όπου χρησιμοποιήσαμε ``collate`` για εκπαίδευση](https://colab.research.google.com/drive/1PAKIUh11Fb-16qZe1lgo8y71ufIbBIVe?usp=sharing))

In [ ]:
tokenized_poem = tokenizer(poem)
tokenized_poem

{'input_ids': [101, 1037, 2944, 10229, 2013, 2951, 1010, 2240, 2011, 2240, 1010, 19158, 15871, 2127, 1996, 7060, 12342, 1012, 17978, 2015, 18855, 2083, 9014, 2784, 1010, 2096, 3279, 10543, 2991, 1998, 15698, 2562, 1012, 3086, 4641, 13594, 2187, 1998, 2157, 1010, 17181, 3574, 1010, 6123, 1010, 2422, 1012, 2013, 19204, 2015, 20288, 2000, 19019, 9742, 1010, 1996, 2897, 16473, 2049, 2219, 2204, 3168, 1012, 2025, 3894, 1517, 2074, 20204, 2012, 4094, 1010, 2664, 2653, 29037, 3458, 1996, 5122, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [ ]:
len(tokenized_poem.attention_mask)

79

In [ ]:
tokenizer(poem, padding='max_length', truncation=True, max_length=128)

{'input_ids': [101, 1037, 2944, 10229, 2013, 2951, 1010, 2240, 2011, 2240, 1010, 19158, 15871, 2127, 1996, 7060, 12342, 1012, 17978, 2015, 18855, 2083, 9014, 2784, 1010, 2096, 3279, 10543, 2991, 1998, 15698, 2562, 1012, 3086, 4641, 13594, 2187, 1998, 2157, 1010, 17181, 3574, 1010, 6123, 1010, 2422, 1012, 2013, 19204, 2015, 20288, 2000, 19019, 9742, 1010, 1996, 2897, 16473, 2049, 2219, 2204, 3168, 1012, 2025, 3894, 1517, 2074, 20204, 2012, 4094, 1010, 2664, 2653, 29037, 3458, 1996, 5122, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
print(len(tokenizer(poem,padding='max_length', truncation=True, max_length=128).input_ids))



128


### Μέθοδος 1 — `SentenceTransformer`

In [ ]:
st_embedding = st_model.encode(poem, normalize_embeddings=True)

print('Type  :', type(st_embedding))
print('Shape :', st_embedding.shape)
print('First 20 values:', st_embedding[:20])

Type  : <class 'numpy.ndarray'>
Shape : (768,)
First 20 values: [-0.00234648 -0.00213542  0.06033161  0.00793328  0.02878368  0.01632576
  0.00205444  0.01618173 -0.00391229 -0.03589844  0.02594078 -0.01818319
 -0.02692685  0.04798818 -0.02298571  0.04532281  0.04702563  0.00499253
 -0.01916536  0.03148982]


### Μέθοδος 2 — `BertEmbedder` (tokenize → forward → pool)

In [ ]:
# Step 1: tokenize
encoded = tokenizer(poem, return_tensors='pt', truncation=True)
input_ids      = encoded['input_ids'].to(device)
attention_mask = encoded['attention_mask'].to(device)

print('Token count (including [CLS] and [SEP]):', input_ids.shape[1])

# Step 2: forward pass + mean pool
with torch.no_grad():
    our_embedding = our_model(input_ids=input_ids, attention_mask=attention_mask)

# Step 3: L2 normalize (same as normalize_embeddings=True in SentenceTransformer)
our_embedding = torch.nn.functional.normalize(our_embedding, p=2, dim=1)
our_embedding = our_embedding[0].cpu().numpy()   # drop the batch dimension

print('Shape :', our_embedding.shape)
print('First 20 values:', our_embedding[:20])

Token count (including [CLS] and [SEP]): 79
Shape : (768,)
First 20 values: [-0.00234648 -0.00213542  0.06033161  0.00793328  0.02878368  0.01632576
  0.00205444  0.01618173 -0.00391229 -0.03589844  0.02594078 -0.01818319
 -0.02692685  0.04798818 -0.02298571  0.04532281  0.04702563  0.00499253
 -0.01916536  0.03148982]


## Είναι τα Δύο Διανύσματα Πανομοιότυπα;

Αν και οι δύο μέθοδοι υλοποιούν τον ίδιο υπολογισμό, τα διανύσματα θα πρέπει να συμφωνούν σε ακρίβεια κινητής υποδιαστολής.

In [ ]:
max_diff = np.abs(st_embedding - our_embedding).max()
mean_diff = np.abs(st_embedding - our_embedding).mean()
cosine_sim = float(np.dot(st_embedding, our_embedding))   # both L2-normalised

print(f'Max absolute difference  : {max_diff:.2e}')
print(f'Mean absolute difference : {mean_diff:.2e}')
print(f'Cosine similarity        : {cosine_sim:.10f}')
print()
print('Vectors match (tol=1e-5):', np.allclose(st_embedding, our_embedding, atol=1e-5))

Max absolute difference  : 0.00e+00
Mean absolute difference : 0.00e+00
Cosine similarity        : 1.0000001192

Vectors match (tol=1e-5): True


Οποιαδήποτε διαφορά είναι σφάλμα στρογγυλοποίησης κινητής υποδιαστολής (< 1e-5). Η ομοιότητα cosine είναι 1.0 ως 10 δεκαδικά ψηφία — τα δύο διανύσματα είναι πανομοιότυπα.

## Σύνοψη

| Βήμα | Κώδικας |
|------|---------|
| Φόρτωση βαρών | `AutoModelForMaskedLM.from_pretrained(...).bert` |
| Tokenization | `AutoTokenizer(...)(text, return_tensors='pt')` |
| Forward pass | `bert(input_ids=..., attention_mask=...)` → `last_hidden_state` |
| Pooling tokens | σταθμισμένος μέσος όρος ανά τη διάσταση ακολουθίας, αγνοώντας το padding |
| Κανονικοποίηση | `F.normalize(emb, p=2, dim=1)` για ομοιότητα cosine μέσω εσωτερικού γινομένου |

Το `SentenceTransformer('bert-base-uncased')` εκτελεί ακριβώς αυτά τα βήματα· εδώ κάναμε το καθένα ρητό.